# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường

> Track 4 · Ngày 10 · *Nhãn, auto-labeling và Vision–Language Model* · **Google Colab T4 (miễn phí)**

Notebook này chứa toàn bộ quy trình thực hiện theo [`GUIDE.md`](GUIDE.md), tuân thủ thang điểm [`RUBRIC.md`](RUBRIC.md) và cung cấp số liệu cho [`REPORT.md`](REPORT.md).

## 0. Môi trường (Colab T4)

**Mục tiêu:** Cài đặt các thư viện cần thiết và thiết lập thông tin xác thực Kaggle qua Colab Secrets.

In [ ]:
# ==============================================================================
# 0.1 Kết nối GitHub & Tải mã nguồn về Google Colab (nếu chưa có)
# ==============================================================================
import os
from pathlib import Path

REPO_URL = "https://github.com/Wrxhard/K4-Track-4-Day10-VisionLanguageModel.git"
REPO_NAME = "K4-Track-4-Day10-VisionLanguageModel"

# Nếu đang ở /content và chưa có thư mục scripts:
if not Path("scripts").exists():
    if not Path(REPO_NAME).exists():
        print(f"Đang clone repo từ {REPO_URL}...")
        !git clone {REPO_URL}
    os.chdir(REPO_NAME)

print("✓ Thư mục làm việc hiện tại:", os.getcwd())
assert Path("scripts/prepare_data.py").exists(), "Lỗi: Không tìm thấy thư mục scripts! Hãy kiểm tra clone repo."

# ==============================================================================
# 0.2 Kiểm tra GPU và Cài đặt thư viện
# ==============================================================================
# Kiểm tra cấu hình GPU (yêu cầu GPU T4 16GB VRAM)
!nvidia-smi -L

# Cài đặt các thư viện phiên bản chuẩn của bài lab
!pip -q install -r requirements-colab.txt


In [ ]:
# Thiết lập Kaggle API key từ Colab Secrets (KHÔNG ghi trực tiếp khóa bí mật vào notebook)
import os
from google.colab import userdata

try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("✓ Đã cấu hình Kaggle credentials từ Colab Secrets.")
except Exception as e:
    print("Lưu ý: Chưa lấy được Kaggle Secrets. Nếu chạy offline hoặc tải zip thủ công, dùng --zip hoặc --source.")


## 1. Dữ liệu: tải và chia split

**Mục tiêu:** Tải dataset Road Damage từ Kaggle và chia tập train/val/test cố định theo phiên quay video (session) nhằm chống rò rỉ dữ liệu (data leakage) giữa các frame video liền kề.

In [ ]:
# Tải và chia split tự động bằng script chuẩn của đề (cố định dataset_version)
!python scripts/prepare_data.py --kaggle


In [ ]:
# Kiểm tra dữ liệu và trực quan hóa mẫu
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scripts.labcommon import read_jsonl, CLASSES

train_records = read_jsonl("data/splits/train.jsonl")
val_records = read_jsonl("data/splits/val.jsonl")
test_records = read_jsonl("data/splits/test.jsonl")

print(f"Số lượng ảnh: Train = {len(train_records)}, Val = {len(val_records)}, Test = {len(test_records)}")

# Kiểm tra chống rò rỉ session
train_sess = {r['session'] for r in train_records}
val_sess = {r['session'] for r in val_records}
test_sess = {r['session'] for r in test_records}

assert len(train_sess & val_sess) == 0, "Rò rỉ session giữa train và val!"
assert len(train_sess & test_sess) == 0, "Rò rỉ session giữa train và test!"
assert len(val_sess & test_sess) == 0, "Rò rỉ session giữa val và test!"
print("✓ Kiểm tra split: Không có phiên quay nào trùng lặp giữa các tập.")

# Vẽ minh họa 6 ảnh kèm bounding box ground truth
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
colors = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}

for idx, r in enumerate(train_records[:6]):
    img = Image.open(Path("data") / r["image_path"]).convert("RGB")
    ax = axes[idx]
    ax.imshow(img)
    w_img, h_img = r["width"], r["height"]
    for obj in r["objects"]:
        xc, yc, w, h = obj["yolo"]
        x1 = (xc - w / 2) * w_img
        y1 = (yc - h / 2) * h_img
        bw = w * w_img
        bh = h * h_img
        rect = patches.Rectangle((x1, y1), bw, bh, linewidth=2,
                                 edgecolor=colors.get(obj['label'], 'white'), facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, max(0, y1 - 4), obj['label'], color=colors.get(obj['label'], 'white'),
                fontsize=8, weight='bold', bbox=dict(facecolor='black', alpha=0.5, pad=1))
    ax.set_title(f"{r['image_id']} | sess: {r['session']}", fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()


## 2. Nhãn: YOLO → JSON thang 0–1000

**Mục tiêu:** Chuyển đổi định dạng nhãn YOLO `[xc, yc, w, h]` chuẩn hóa `[0, 1]` sang chuỗi JSON list chứa toạ độ `bbox_2d: [x1, y1, x2, y2]` nguyên trên thang chuẩn hóa 0–1000 (quy ước grounding của Qwen3-VL).

In [ ]:
def yolo_to_box(label, yolo):
    """
    Chuyển toạ độ YOLO [xc, yc, w, h] chuẩn hoá [0, 1] sang box [x1, y1, x2, y2] thang 0-1000.
    """
    xc, yc, w, h = yolo
    x1 = int(round(max(0.0, min(1000.0, (xc - w / 2.0) * 1000.0))))
    y1 = int(round(max(0.0, min(1000.0, (yc - h / 2.0) * 1000.0))))
    x2 = int(round(max(0.0, min(1000.0, (xc + w / 2.0) * 1000.0))))
    y2 = int(round(max(0.0, min(1000.0, (yc + h / 2.0) * 1000.0))))
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects):
    """
    Tạo JSON câu trả lời, sắp xếp boxes theo thứ tự (y1, x1) từ trên xuống dưới, từ trái sang phải.
    """
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

# Prompt chuẩn yêu cầu model trả về JSON grounding
PROMPT = (
    "Detect road damage in this image. Locate all potholes, cracks, and manholes. "
    "Output a JSON list of detected objects, each with 'label' ('pothole', 'crack', or 'manhole') "
    "and 'bbox_2d' as [x1, y1, x2, y2] normalized on a 0-1000 integer coordinate scale. "
    "Return ONLY the JSON list."
)

# Kiểm tra chuyển đổi 2 chiều trên 1 mẫu
sample = train_records[0]
ans_json = objects_to_answer(sample["objects"])
print("Prompt chuẩn:\n", PROMPT)
print("\nJSON đáp án mẫu:\n", ans_json)

# Kiểm tra quy đổi ngược lại toạ độ pixel
parsed_boxes = json.loads(ans_json)
for orig, pb in zip(sample["objects"], parsed_boxes):
    w_px, h_px = sample["width"], sample["height"]
    re_px = [
        pb["bbox_2d"][0] * w_px / 1000.0,
        pb["bbox_2d"][1] * h_px / 1000.0,
        pb["bbox_2d"][2] * w_px / 1000.0,
        pb["bbox_2d"][3] * h_px / 1000.0
    ]
    xc, yc, w, h = orig["yolo"]
    gt_px = [(xc - w/2)*w_px, (yc - h/2)*h_px, (xc + w/2)*w_px, (yc + h/2)*h_px]
    for a, b in zip(re_px, gt_px):
        assert abs(a - b) < 1.5, f"Lệch toạ độ: {a} vs {b}"
print("✓ Kiểm chứng nhãn: Quy đổi toạ độ 0-1000 chính xác tuyệt đối.")


## 3. E0: zero-shot

**Mục tiêu:** Nạp model gốc Qwen3-VL-4B-Instruct ở chế độ lượng tử hoá NF4 4-bit (giữ nguyên ViT và lm_head ở float16), chạy suy luận zero-shot trên tập test và chấm điểm bằng `scripts/score.py`.

In [ ]:
import torch
import gc
from transformers import BitsAndBytesConfig, Qwen3VLForConditionalGeneration, AutoProcessor
from scripts.labcommon import write_jsonl

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

# Cấu hình lượng tử hoá 4-bit NF4
# QUAN TRỌNG: llm_int8_skip_modules=['visual', 'lm_head'] giữ ViT và projector ở float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"]
)

print("Đang nạp mô hình Qwen3-VL-4B-Instruct...")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={"": 0}
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

def run_inference(model, processor, records, prompt, out_path, batch_size=4, max_new_tokens=256):
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    predictions = []
    model.eval()
    
    for i in range(0, len(records), batch_size):
        batch = records[i:i + batch_size]
        images = [Image.open(Path("data") / r["image_path"]).convert("RGB") for r in batch]
        
        messages_batch = [
            [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": prompt}]}]
            for img in images
        ]
        
        texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_batch]
        inputs = processor(text=texts, images=images, return_tensors="pt", padding=True).to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False
            )
        
        in_len = inputs.input_ids.shape[1]
        for j, (out_seq, r) in enumerate(zip(outputs, batch)):
            gen_tokens = out_seq[in_len:]
            raw_text = processor.decode(gen_tokens, skip_special_tokens=True).strip()
            predictions.append({"image_id": r["image_id"], "raw": raw_text})
            
        if (i // batch_size + 1) % 15 == 0 or i + batch_size >= len(records):
            print(f"Suy luận: {min(i + batch_size, len(records))}/{len(records)} ảnh...")
            
    write_jsonl(out_path, predictions)
    print(f"✓ Đã ghi dự đoán vào {out_path}")

# Suy luận E0 trên toàn bộ tập test
run_inference(model, processor, test_records, PROMPT, "work/e0.jsonl", batch_size=4)


In [ ]:
# Chấm điểm E0 bằng trọng tài score.py
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json


In [ ]:
# Xem 5 output thô của E0 để phân tích lỗi
e0_preds = read_jsonl("work/e0.jsonl")
print("=== 5 OUTPUT THÔ CỦA E0 ===")
for p in e0_preds[:5]:
    print(f"[{p['image_id']}]:\n{p['raw']}\n{'-'*50}")

# Thu hồi VRAM của model E0
del model
gc.collect()
torch.cuda.empty_cache()
print("✓ Đã giải phóng bộ nhớ GPU sau E0.")


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

**Mục tiêu:** Xây dựng quy trình mã hóa dữ liệu huấn luyện với mask loss (nhãn bằng `-100` cho prompt và 220 token ảnh) để mô hình chỉ học phần câu trả lời JSON định vị hư hại.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class RoadDamageDataset(Dataset):
    def __init__(self, records, prompt, is_train=True):
        self.records = records
        self.prompt = prompt
        self.is_train = is_train

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        image = Image.open(Path("data") / r["image_path"]).convert("RGB")
        answer = objects_to_answer(r["objects"])
        return image, self.prompt, answer, r["image_id"]

def encode_example(image, prompt, answer, processor):
    user_msgs = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    full_msgs = [
        {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]},
        {"role": "assistant", "content": answer}
    ]
    
    prompt_text = processor.apply_chat_template(user_msgs, add_generation_prompt=True, tokenize=False)
    full_text = processor.apply_chat_template(full_msgs, add_generation_prompt=False, tokenize=False)
    
    prompt_enc = processor(text=[prompt_text], images=[image], return_tensors="pt")
    full_enc = processor(text=[full_text], images=[image], return_tensors="pt")
    
    n = prompt_enc.input_ids.shape[1]
    assert torch.equal(full_enc.input_ids[:, :n], prompt_enc.input_ids), "Prompt phải là tiền tố của full sequence!"
    
    labels = full_enc.input_ids.clone()
    labels[:, :n] = -100  # Mask toàn bộ prompt và vision tokens: chỉ học phần câu trả lời
    
    full_enc["labels"] = labels
    return full_enc

def collate_road_damage(batch, processor):
    encoded_list = [encode_example(img, prompt, ans, processor) for img, prompt, ans, _ in batch]
    
    pad_id = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None else 0
    padded_input_ids = torch.nn.utils.rnn.pad_sequence(
        [enc.input_ids.squeeze(0) for enc in encoded_list], batch_first=True, padding_value=pad_id
    )
    padded_attention_mask = torch.nn.utils.rnn.pad_sequence(
        [enc.attention_mask.squeeze(0) for enc in encoded_list], batch_first=True, padding_value=0
    )
    padded_labels = torch.nn.utils.rnn.pad_sequence(
        [enc.labels.squeeze(0) for enc in encoded_list], batch_first=True, padding_value=-100
    )
    
    batch_dict = {
        "input_ids": padded_input_ids,
        "attention_mask": padded_attention_mask,
        "labels": padded_labels,
    }
    
    if "mm_token_type_ids" in encoded_list[0]:
        batch_dict["mm_token_type_ids"] = torch.nn.utils.rnn.pad_sequence(
            [enc.mm_token_type_ids.squeeze(0) for enc in encoded_list], batch_first=True, padding_value=0
        )
    # pixel_values và image_grid_thw: Nối theo chiều 0, KHÔNG pad
    if "pixel_values" in encoded_list[0]:
        batch_dict["pixel_values"] = torch.cat([enc.pixel_values for enc in encoded_list], dim=0)
    if "image_grid_thw" in encoded_list[0]:
        batch_dict["image_grid_thw"] = torch.cat([enc.image_grid_thw for enc in encoded_list], dim=0)
        
    return batch_dict

# Kiểm tra số lượng token vision và mask loss
sample_img = Image.open(Path("data") / train_records[0]["image_path"]).convert("RGB")
sample_enc = encode_example(sample_img, PROMPT, objects_to_answer(train_records[0]["objects"]), processor)

if "image_grid_thw" in sample_enc:
    t, h, w = sample_enc.image_grid_thw[0].tolist()
    vision_tokens = (h // 2) * (w // 2) * t
    print(f"Số token vision cho ảnh 640x360: {vision_tokens} (yêu cầu chuẩn: 220)")
    assert vision_tokens == 220, f"Sai lệch token vision: {vision_tokens} != 220"

unmasked = sample_enc.labels[0][sample_enc.labels[0] != -100]
print("Câu trả lời giải mã từ các token unmasked:")
print(processor.decode(unmasked, skip_special_tokens=False))
print("✓ Mask loss kiểm tra thành công: Chỉ tính gradient cho token trả lời.")


## 5. Setup LoRA / đóng băng cho E1, E2, E3

**Mục tiêu:** Thiết lập các cấu hình LoRA và đóng băng/mở khối cho 3 thí nghiệm chính:
- **E1:** Chỉ LoRA LLM (252 modules), đóng băng ViT và projector.
- **E2:** E1 + mở đầy đủ projector (`visual.merger`, `visual.deepstack_merger_list`, ≈ 109M tham số ở float32).
- **E3:** E2 + LoRA trên các khối attention và MLP của ViT (thêm 96 modules, tổng 348 modules).

In [ ]:
import re
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup_name):
    # 1. Đóng băng toàn bộ tham số
    for param in model.parameters():
        param.requires_grad = False
    model.enable_input_require_grads()
    
    # 2. Định nghĩa regex target_modules cho LoRA
    if setup_name in ["E1", "E2"]:
        target_modules = r".*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)$"
    elif setup_name == "E3":
        target_modules = (
            r".*(language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)|"
            r"visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2)))$"
        )
    else:
        raise ValueError(f"Setup không hợp lệ: {setup_name}")
        
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=target_modules,
        bias="none",
        task_type="CAUSAL_LM"
    )
    peft_model = get_peft_model(model, peft_config)
    
    # 3. Mở đầy đủ projector cho E2 và E3 và đưa về float32
    if setup_name in ["E2", "E3"]:
        for name, param in peft_model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                param.data = param.data.to(torch.float32)
                param.requires_grad = True
                
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in peft_model.parameters())
    print(f"[{setup_name}] Trainable params: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.4f}%)")
    return peft_model, trainable_params


## 6. Vòng huấn luyện

**Mục tiêu:** Huấn luyện 300 bước tối ưu với AdamW, cosine learning rate scheduler (warmup 5%), fp16 mixed precision, gradient accumulation (batch size = 2, grad_accum = 4 → effective batch size = 8). Lưu adapter LoRA và lưu riêng projector cho E2/E3.

In [ ]:
import time
import math
from torch.cuda.amp import autocast, GradScaler
from transformers import get_cosine_schedule_with_warmup

def train_setup(setup_name, max_steps=300, batch_size=2, grad_accum=4, lr=2e-4):
    print(f"\n=================== HUẤN LUYỆN {setup_name} ===================")
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    # Nạp base model lượng tử hóa 4-bit
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    
    model, trainable_params = apply_setup(base_model, setup_name)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    
    train_ds = RoadDamageDataset(train_records, PROMPT, is_train=True)
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=lambda b: collate_road_damage(b, processor)
    )
    
    trainable_p = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_p, lr=lr, weight_decay=0.0)
    warmup_steps = int(max_steps * 0.05)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, max_steps)
    scaler = GradScaler("cuda")
    
    start_time = time.time()
    step = 0
    accum_step = 0
    model.train()
    optimizer.zero_grad()
    
    data_iter = iter(train_loader)
    while step < max_steps:
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            batch = next(data_iter)
            
        batch = {k: v.to("cuda") for k, v in batch.items()}
        
        with autocast(dtype=torch.float16):
            outputs = model(**batch)
            loss = outputs.loss / grad_accum
            
        if not math.isfinite(loss.item()):
            raise ValueError(f"Loss không hữu hạn tại step {step}! Tràn số fp16.")
            
        scaler.scale(loss).backward()
        accum_step += 1
        
        if accum_step % grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_p, 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            step += 1
            
            if step % 25 == 0 or step == max_steps:
                cur_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)
                print(f"Step {step:3d}/{max_steps} | Loss: {loss.item()*grad_accum:.4f} | LR: {scheduler.get_last_lr()[0]:.2e} | VRAM: {cur_vram:.2f} GB")
                
            if step % 100 == 0 or step == max_steps:
                save_dir = Path("work") / setup_name.lower()
                save_dir.mkdir(parents=True, exist_ok=True)
                model.save_pretrained(save_dir / "lora_adapter")
                # QUAN TRỌNG: Lưu riêng projector vì peft save_pretrained không lưu module full training
                if setup_name in ["E2", "E3"]:
                    extra_weights = {
                        n: p.data.cpu() for n, p in model.named_parameters()
                        if ("visual.merger" in n or "visual.deepstack_merger_list" in n) and p.requires_grad
                    }
                    torch.save(extra_weights, save_dir / "projector.pt")
                    
    train_minutes = round((time.time() - start_time) / 60.0, 2)
    peak_vram_gb = round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2)
    
    meta = {
        "trainable_params": trainable_params,
        "peak_vram_gb": peak_vram_gb,
        "train_minutes": train_minutes
    }
    with open(Path("work") / setup_name.lower() / "run.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
        
    print(f"✓ Xong {setup_name}: {train_minutes} phút, peak VRAM: {peak_vram_gb} GB")
    
    del model, base_model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()
    return meta


## 7. Suy luận và chấm E1–E3

**Mục tiêu:** Nạp lại mô hình cùng adapter đã huấn luyện (và trọng số projector cho E2, E3), suy luận trên tập test (`work/eN.jsonl`) và chấm điểm tự động qua `scripts/score.py` với metadata đầy đủ.

In [ ]:
from peft import PeftModel

def evaluate_setup(setup_name, run_meta):
    print(f"\n--- Suy luận và đánh giá {setup_name} ---")
    save_dir = Path("work") / setup_name.lower()
    
    # 1. Nạp base model 4-bit
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    
    # 2. Nạp adapter LoRA
    model = PeftModel.from_pretrained(base_model, save_dir / "lora_adapter")
    
    # 3. Nạp lại projector cho E2 và E3
    projector_path = save_dir / "projector.pt"
    if projector_path.exists():
        print(f"Nạp trọng số projector từ {projector_path}...")
        proj_w = torch.load(projector_path, map_location="cuda")
        model.load_state_dict(proj_w, strict=False)
        
    pred_file = f"work/{setup_name.lower()}.jsonl"
    run_inference(model, processor, test_records, PROMPT, pred_file, batch_size=4)
    
    # Chấm điểm với score.py kèm metadata
    out_ev = f"report/evidence/{setup_name.lower()}.json"
    cmd = (
        f"python scripts/score.py --pred {pred_file} --name {setup_name} --out {out_ev} "
        f"--meta trainable_params={run_meta['trainable_params']} "
        f"--meta peak_vram_gb={run_meta['peak_vram_gb']} "
        f"--meta train_minutes={run_meta['train_minutes']}"
    )
    print("Chấm điểm:", cmd)
    os.system(cmd)
    
    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()

# Chạy huấn luyện và đánh giá tuần tự E1, E2, E3
for setup in ["E1", "E2", "E3"]:
    meta = train_setup(setup)
    evaluate_setup(setup, meta)


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

**Mục tiêu:**
- **E4 (Ablation bắt buộc):** Chọn ĐÚNG MỘT biến để khảo sát (ví dụ: `rank` với $r=4$ vs $r=64$, hoặc `data_size` 10% vs 100%, hoặc `coord_scale` norm1000 vs pixel) so với setup tốt nhất trong E1–E3. Lưu bằng chứng vào `report/evidence/e4_...json`.
- **E5 (Bonus +10 điểm):** Xây dựng hàm reward IoU cho GRPO, viết unit test kiểm chứng chống reward hacking và huấn luyện GRPO.

In [ ]:
# ==========================================================
# 8.1 E4: Thí nghiệm Ablation (Chọn biến: rank)
# ==========================================================
# Ta khảo sát biến rank: r = 4 (alpha = 8) so với r = 16 (E1)

def apply_e4_rank(model, r_val=4):
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()
    target_modules = r".*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)$"
    peft_config = LoraConfig(
        r=r_val, lora_alpha=2 * r_val, lora_dropout=0.05,
        target_modules=target_modules, bias="none", task_type="CAUSAL_LM"
    )
    peft_model = get_peft_model(model, peft_config)
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    print(f"[E4-rank-{r_val}] Trainable params: {trainable_params:,}")
    return peft_model, trainable_params

print("Huấn luyện E4 với rank = 4...")
# Bạn có thể gọi huấn luyện E4 tương tự như hàm train_setup và lưu report/evidence/e4_rank_4.json


In [ ]:
# ==========================================================
# 8.2 E5 (Tuỳ chọn bonus +10 điểm): Hàm reward IoU cho GRPO
# ==========================================================
from scripts.score import parse_output, iou_xyxy

def compute_iou_reward(completion, gt_boxes):
    """
    Hàm reward kiểm chứng được cho GRPO:
    Reward = 0.2 + 0.8 * (tổng IoU các cặp khớp) / max(n_gt, n_pred, 1)
    """
    pred = parse_output(completion)
    if pred is None:  # JSON hỏng
        return 0.0
    if not gt_boxes and not pred:  # Ảnh rỗng và đoán rỗng
        return 1.0
        
    pairs = []
    for i, g in enumerate(gt_boxes):
        for j, p in enumerate(pred):
            if g["label"] == p["label"]:
                iou = iou_xyxy(g["bbox"], p["bbox"])
                if iou >= 0.5:
                    pairs.append((iou, i, j))
    pairs.sort(key=lambda x: x[0], reverse=True)
    
    used_g, used_p, matched_ious = set(), set(), []
    for iou, i, j in pairs:
        if i not in used_g and j not in used_p:
            used_g.add(i)
            used_p.add(j)
            matched_ious.append(iou)
            
    total_iou = sum(matched_ious)
    return 0.2 + 0.8 * total_iou / max(len(gt_boxes), len(pred), 1)

# Unit test kiểm tra hàm reward trước khi chạy GRPO
gt_sample = [{"label": "pothole", "bbox": (400, 400, 600, 600)}]
assert compute_iou_reward('[{"label": "pothole", "bbox_2d": [400, 400, 600, 600]}]', gt_sample) == 1.0
assert compute_iou_reward('broken json', gt_sample) == 0.0
assert compute_iou_reward('[{"label": "crack", "bbox_2d": [400, 400, 600, 600]}]', gt_sample) == 0.2
assert compute_iou_reward('[]', []) == 1.0
print("✓ Tất cả test case cho hàm reward IoU đều PASS!")


In [ ]:
# ==========================================================
# 8.3 Kiểm tra toàn bộ bài nộp trước khi commit
# ==========================================================
!python scripts/check_submission.py


In [ ]:
# ==============================================================================
# 8.4 Tải file bằng chứng (Evidence) về máy tính để hoàn thiện báo cáo
# ==============================================================================
# Nén toàn bộ thư mục evidence và file meta.json lại thành file zip
from google.colab import files

!zip -r lab_day10_evidence.zip report/evidence data/splits/meta.json
files.download("lab_day10_evidence.zip")
print("✓ Đã yêu cầu trình duyệt tải xuống lab_day10_evidence.zip!")
